In [20]:
!pip install -q llama-index-core llama-index-embeddings-huggingface llama-index-vector-stores-chroma llama-index-retrievers-bm25 sentence-transformers chromadb pypdf requests fonttools


In [21]:
PDF_DIR  = "../uploads"
CHROMA_DB_PATH = "./chroma_db/rag_docs"
COLLECTION_NAME = "rag_docs"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 200
TOP_K = 10
API_KEY = "your-openrouter-api-key"
MODEL_NAME = "openai/gpt-4o-mini"
MAX_TOKENS = 4096

In [22]:
from pathlib import Path
from pypdf import PdfReader
from llama_index.core import Document

pdf_path = Path(PDF_DIR)
pdf_files = list(pdf_path.glob("*.pdf"))
raw_document = []
for pdf_file in pdf_files:
    try:
        reader = PdfReader(str(pdf_file))
        total_pages = len(reader.pages)

        print(f"{pdf_file.name} - {total_pages}")

        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if not text.strip():
                continue
            raw_document.append(Document(
                text = text,
                metadata = {
                    "source":pdf_file.name,
                    "page_num": page_num +1,
                    "total_pages": total_pages
                }
            ))
    except Exception as E:
        print("Document cannot be loaded due to", E)

dl_full_notes.pdf - 128
dl_sample_notes.pdf - 3


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/Univers-LightUltraCondensed', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 0, '/FontDescriptor': IndirectObject(11479, 0, 2632535620144), '/LastChar': 255, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 138, 258, 278, 276, 276, 480, 370, 168, 167, 167, 278, 600, 138, 166, 138, 278, 276, 276, 276, 276, 276, 276, 276, 276, 276, 276, 138, 138, 600, 600, 600, 278, 800, 332, 333, 278, 333, 259, 259, 333, 333, 168, 278, 278, 259, 444, 333, 334, 278, 334, 333, 278, 260, 334, 278, 444, 278, 278, 259, 167, 278, 167, 600, 500, 148, 278, 278, 259, 278, 278, 167, 278, 278, 148, 148, 278, 148, 406, 278, 278, 278, 278, 204, 259, 167, 278, 258, 406, 258, 259, 204, 167, 222, 167, 600, 500, 138, 500, 168, 276, 278, 1000, 278, 278, 148,

mml-book.pdf - 417


Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.
Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.
Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


In [23]:
raw_document[0]

Document(id_='9ec3183c-cdaf-45a2-b2a1-5c8157848f74', embedding=None, metadata={'source': 'dl_sample_notes.pdf', 'page_num': 1, 'total_pages': 3}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='Unit 1: Deep Learning BasicsDeep learning is a subset of machine learning based on artificial neural networks with representation learning.Key topics include:- Linear Algebra: Scalars, Vectors, Matrices, and Tensors.- Probability Distributions and Maximum Likelihood Estimation.- Gradient-based optimization: Stochastic Gradient Descent (SGD) and Adam optimizer.- Hyperparameters: learning rate, batch size, epochs, and regularization techniques.', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}')

In [24]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size = CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP
)

raw_chunk = splitter.get_nodes_from_documents(raw_document)

In [25]:
raw_chunk[11]

TextNode(id_='a8426f00-6b1a-4231-8565-a68e7f629b86', embedding=None, metadata={'source': 'mml-book.pdf', 'page_num': 11, 'total_pages': 417}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='fb7e4fa4-63ba-465c-ac40-7a258b993177', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'source': 'mml-book.pdf', 'page_num': 11, 'total_pages': 417}, hash='26150aa84a152552ca2399251cfa23175297e7b4472af410d0db36e32a6328fe')}, metadata_template='{key}: {value}', metadata_separator='\n', text='Foreword 5\nMaximus McCann\nMengyan Zhang\nMichael Bennett\nMichael Pedersen\nMinjeong Shin\nMohammad Malekzadeh\nNaveen Kumar\nNico Montali\nOscar Armas\nPatrick Henriksen\nPatrick Wieschollek\nPattarawat Chormai\nPaul Kelly\nPetros Christodoulou\nPiotr Januszewski\nPranav Subramani\nQuyu Kong\nRagib Zaman\nRui Zhang\nRyan-Rhys Griffiths\nSalomon Kabongo\nSamuel Ogunmola\nSandeep Mavadia\nSarvesh Nikumbh\nSebastian Raschka\nS

In [26]:
def clean_chunk(text):
    text = str(text.encode('utf-8', errors = 'ignore').decode('utf-8'))
    return "".join(text.strip())

chunk = []
for node in raw_chunk:
    content = node.get_content().strip()
    if content:
        node.text = clean_chunk(content)
        chunk.append(node)

In [27]:
chunk[18]

TextNode(id_='7bf18ca7-72e2-4af9-b2f4-a36c6603dedd', embedding=None, metadata={'source': 'mml-book.pdf', 'page_num': 20, 'total_pages': 417}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='b94a2d50-5784-4eeb-bed2-1431edb470e4', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'source': 'mml-book.pdf', 'page_num': 20, 'total_pages': 417}, hash='5bfa2a67d87b67a87868e19c755a3f2a71d779810fc8871b326fd6deb261fd3a')}, metadata_template='{key}: {value}', metadata_separator='\n', text='14 Introduction and Motivation\nFigure 1.1 The\nfoundations and\nfour pillars of\nmachine learning.\nClassiﬁcation \nDensity \nEstimation \nRegression \nDimensionality \nReduction \nMachine Learning\nVector Calculus Probability & Distributions Optimization\nAnalytic Geometry Matrix DecompositionLinear Algebra\nbetween the two parts of the book to link mathematical concepts with\nmachine learning algorithms.\nOf course there ar

In [28]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embedding = HuggingFaceEmbedding(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

print("model_loaded_sucessfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

model_loaded_sucessfully


In [29]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

chroma_db = Path(CHROMA_DB_PATH)

chroma_client = chromadb.PersistentClient(path = chroma_db)
chroma_collection = chroma_client.get_or_create_collection(COLLECTION_NAME)
vector_store = ChromaVectorStore(chroma_collection = chroma_collection)

index = VectorStoreIndex.from_vector_store(
    vector_store = vector_store,
    embed_model = embedding
)
if not chunk:
    print("No chunks to loaded into vector_db")
index.insert_nodes(chunk)
print(chroma_collection.count())

422


In [32]:
import requests

class OpenRouterLLM:

    def __init__(self, api_key: str, model: str =MODEL_NAME, max_token: int = MAX_TOKENS ):
        self.api_key  = api_key
        self.model = model
        self.max_token = max_token

    def invoke(self, prompt):

        response = requests.post(
            url = "https://openrouter.ai/api/v1/chat/completions",
            headers = {
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json"
            },
            json = {
                "model": self.model,
                "max_tokens": self.max_token,
                "messages": [
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            },
            timeout = 60
        )

        response.raise_for_status()

        data = response.json()

        return data["choices"][0]["message"]["content"]

LLM = OpenRouterLLM(api_key = API_KEY)
LLM.invoke("Who is the president of india")


'As of my last update in October 2023, the President of India is Droupadi Murmu. She took office on July 25, 2022, becoming the first tribal woman to hold the position. Please verify with current sources, as political positions can change.'

In [33]:
Prompt_Template = """
You are an expert study assistant. Using ONLY the context provided below,
give a thorough and detailed answer to the question.

Your answer should:
- Be comprehensive and well-structured
- Use bullet points or numbered lists where appropriate
- Include relevant examples or explanations from the context
- Use headings to organise if the answer covers multiple topics
- Be as detailed as possible — do NOT summarise briefly
- If the answer is not in the context, say "I don't have enough information."

context : {context},

Question :{question}

Detailed Answer:
"""

from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.llms.mock import MockLLM

# 1. Dense Retriever (ChromaDB + MiniLM)
dense_retriever = index.as_retriever(
    similarity_top_k = TOP_K
)

# 2. Sparse Retriever (BM25 keyword search)
bm25_retriever = BM25Retriever.from_defaults(
    nodes = chunk,
    similarity_top_k = TOP_K
)

# 3. Hybrid Retriever with Reciprocal Rank Fusion (RRF)
hybrid_retriever = QueryFusionRetriever(
    retrievers = [dense_retriever, bm25_retriever],
    similarity_top_k = TOP_K,
    num_queries = 1,
    mode = "reciprocal_rerank",
    use_async = False,
    llm = MockLLM()
)

def format_nodes(nodes):
    parts = []
    for n in nodes:
        source = n.node.metadata.get("source", "unknown")
        page = n.node.metadata.get("page_num", "N/A")
        parts.append(f"[Source: {source} | Page {page}]\n{n.node.get_content()}")
    return "\n\n".join(parts)

print("Hybrid RAG Pipeline (Dense + BM25 via RRF) is ready!")


Hybrid RAG Pipeline (Dense + BM25 via RRF) is ready!


In [35]:
def ask(question: str, mode: str = "hybrid"):
    print(f"\n{'='*60}")
    print(f" Question: {question} (Mode: {mode})")
    print("="*60)
    
    nodes = hybrid_retriever.retrieve(question)

    content_str = format_nodes(nodes)
    prompt = Prompt_Template.format(
        context = content_str,
        question = question
    )

    try:
        answer = LLM.invoke(prompt)
    except Exception as E:
        print("Error:", E)
        return ""

    print("Answer: ", answer)

ask("Explain about Document")



 Question: Explain about Document (Mode: hybrid)
Answer:  # Explanation of Document

The document appears to be a scholarly text on the mathematics underpinning machine learning, particularly focusing on fundamental concepts that guide the development of machine learning algorithms. The content is structured into various sections that address different essential components of machine learning, including data, models, learning paradigms, and relevant mathematical foundations. Below is a detailed breakdown based on various sections of the document.

## 1. Introduction to Machine Learning

- **Purpose**: The document introduces machine learning as a field primarily concerned with designing algorithms that can autonomously extract useful patterns and information from datasets. 
- **Core Concepts**: The three fundamental concepts outlined are:
  - **Data**: Essential for machine learning, as algorithms derive insights from it.
  - **Model**: Represents a mathematical construct that describ